# Efficiently Fine-Tuning a Large Language Model 

This notebook covers two broad approaches for working with Large Language Models (LLMs):

1. **Prompting** — Steering a pre-trained model through carefully designed inputs, without changing any weights.
2. **Fine-Tuning** — Updating model weights on a task-specific dataset to specialise behaviour.

---

## Why Fine-Tune?

Pre-trained LLMs are trained on broad internet text and already possess impressive general knowledge. However, they may:

- Produce output in the wrong format for a downstream system
- Lack domain-specific knowledge (medical, legal, etc.)
- Not follow instructions reliably
- Generate responses in the wrong language or style

Fine-tuning addresses these gaps by training on curated examples.

---

## Fine-Tuning Strategies

### Supervised Fine-Tuning (SFT)
The model is trained directly on `(instruction, response)` pairs. The loss is standard cross-entropy over the response tokens. This is the simplest and most common fine-tuning approach.

### Reinforcement Learning from Human Feedback (RLHF)
Human annotators rank model outputs. A separate *reward model* is trained on these rankings, and the LLM is then optimised to maximise the reward using Proximal Policy Optimisation (PPO). This is how ChatGPT was trained, but it is complex and expensive.

### Direct Preference Optimisation (DPO)
A simpler alternative to RLHF that skips the reward model entirely. The LLM is trained directly on `(prompt, chosen_response, rejected_response)` triples, making it much easier to implement while achieving similar alignment quality.

---

In this notebook we focus on **SFT** using the modern **QLoRA** approach.

## Setup

We need the following libraries:

| Library | Purpose |
|---|---|
| `transformers` | Model loading, tokenisation, generation |
| `peft` | LoRA adapter creation and management |
| `trl` | `SFTTrainer` and `SFTConfig` for instruction fine-tuning |
| `datasets` | Dataset loading from the HuggingFace hub |
| `bitsandbytes` | 4-bit quantisation (BnB) |
| `accelerate` | Multi-GPU / mixed-precision training backend |

> **Note for HPC users**: On SLURM clusters, pre-install these in your virtual environment or Singularity container rather than running pip inside the notebook.

In [ ]:
!pip install -q -U trl transformers accelerate
!pip install -q -U datasets bitsandbytes peft

In [ ]:
# Authenticate to access gated models (e.g. Llama-3)
# You need a HuggingFace account and to accept the model licence at huggingface.co/meta-llama
!huggingface-cli login

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Store weights in 4-bit
    bnb_4bit_quant_type="nf4",      # Use NormalFloat 4-bit (optimal for normally-distributed weights)
    bnb_4bit_use_double_quant=True, # Quantise the quantisation constants too (~0.37 bits saved/param)
    bnb_4bit_compute_dtype=torch.bfloat16,  # Upcast to bfloat16 for actual matrix multiplications
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",       # Automatically distribute layers across available GPUs
    trust_remote_code=True,
)
model.config.use_cache = False  # Disable KV cache during training (not needed, saves memory)

/home/azagar/Documents/NLP-Course-Tutorials/.vllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/azagar/Documents/NLP-Course-Tutorials/.vllm/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()
Loading weights: 100%|██████████| 291/291 [02:17<00:00,  2.12it/s]


In [2]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Llama-3 does not have a dedicated pad token — use eos_token as pad
# padding_side='right' avoids overflow issues in half-precision training
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

---

## Loading the Fine-Tuning Dataset

We use **OpenAssistant Guanaco** — the dataset from the original QLoRA paper. It contains ~10k human-written conversational exchanges in English, already formatted with the `### Human:` / `### Assistant:` template that instruction-tuned models expect.

Each example looks like:
```
### Human: What is the capital of France?
### Assistant: The capital of France is Paris.
```

In a real project you would inspect:
1. **Size** — is there enough data? LoRA fine-tuning can work with as few as a few hundred examples.
2. **Format** — does the text field follow a consistent chat template?
3. **Quality** — are responses the kind of behaviour you want to reinforce?

In [3]:
from datasets import load_dataset

dataset_name = "timdettmers/openassistant-guanaco"
dataset = load_dataset(dataset_name, split="train")

print(f"Dataset size: {len(dataset)} examples")
print(f"Features: {dataset.features}")
print("\nFirst example:")
print(dataset["text"][2])

Repo card metadata block was not found. Setting CardData to empty.


Dataset size: 9846 examples
Features: {'text': Value('string')}

First example:
### Human: Can you explain contrastive learning in machine learning in simple terms for someone new to the field of ML?### Assistant: Sure! Let's say you want to build a model which can distinguish between images of cats and dogs. You gather your dataset, consisting of many cat and dog pictures. Then you put them through a neural net of your choice, which produces some representation for each image, a sequence of numbers like [0.123, 0.045, 0.334, ...]. The problem is, if your model is unfamiliar with cat and dog images, these representations will be quite random. At one time a cat and a dog picture could have very similar representations (their numbers would be close to each other), while at others two cat images may be represented far apart. In simple terms, the model wouldn't be able to tell cats and dogs apart. This is where contrastive learning comes in.

The point of contrastive learning is to take pa

## Parameter-Efficient Fine-Tuning (PEFT)

Full fine-tuning updates **all** model parameters. For a 7B model with the Adam optimiser, this requires:
- **Weights**: ~14 GB (bfloat16)
- **Gradients**: ~14 GB
- **Optimizer states** (Adam keeps 1st and 2nd moments): ~56 GB
- **Total**: ~84 GB — far beyond a typical research GPU

**PEFT** methods freeze most weights and only train a small number of additional parameters, reducing memory requirements by 10–100×.

---

## Low-Rank Adaptation (LoRA)

LoRA is the most widely used PEFT method. The core idea is elegant:

### The Mathematical Intuition

During fine-tuning, the *change* in a weight matrix $\Delta W$ tends to have a **low intrinsic rank** — meaning most of the useful update can be expressed in a much lower-dimensional subspace. LoRA exploits this by:

$$W' = W + \Delta W = W + \frac{\alpha}{r} \cdot B A$$

Where:
- $W \in \mathbb{R}^{d \times k}$ — the frozen original weight matrix
- $A \in \mathbb{R}^{r \times k}$ — a trainable down-projection (randomly initialised)
- $B \in \mathbb{R}^{d \times r}$ — a trainable up-projection (initialised to zeros, so $\Delta W = 0$ at start)
- $r$ — the **rank** (typically 4–64, much smaller than $d$ or $k$)
- $\alpha$ — a scaling hyperparameter

The original weights $W$ are frozen. Only $A$ and $B$ are trained.

### Parameter Savings

For a weight matrix of shape $d \times k$:
- Original parameters: $d \times k$
- LoRA parameters: $r \times k + d \times r = r(d + k)$
- Compression ratio: $\frac{dk}{r(d+k)}$ — typically **10–100×** fewer trainable params

![LoRA diagram](LoRA.png)

### Which Layers to Apply LoRA To?

LoRA can be applied to any linear layer in the transformer. Common choices:

| Layers | Notes |
|---|---|
| `q_proj, v_proj` | Original LoRA paper default — attention query and value projections |
| `q_proj, k_proj, v_proj, o_proj` | All attention projections — better quality, more params |
| `gate_proj, up_proj, down_proj` | Feed-forward layers — important for knowledge tasks |
| All linear layers | Maximum quality, approaches full fine-tuning |

The `target_modules` parameter in `LoraConfig` controls this.

### LoRA Applied to Transformer Blocks

![LoRA Transformer](LoRATransformer.png)

### LoRA Hyperparameters

- **`r` (rank)**: Dimension of the low-rank matrices. Higher rank = more parameters = better expressivity but more memory. Typical range: 8–64.
- **`lora_alpha`**: Scaling factor applied to $BA$. The effective scale is `alpha/r`. Setting `alpha = 2*r` is a common heuristic.
- **`lora_dropout`**: Dropout applied to the LoRA activations. Helps prevent overfitting on small datasets.
- **`bias`**: Whether to train bias terms. Usually left as `"none"` for memory efficiency.
- **`target_modules`**: Which linear layers to apply LoRA to.

In [4]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,                   # Rank: balance between expressivity and parameter count
    lora_alpha=32,          # Scale factor (alpha/r = 2, a common starting point)
    lora_dropout=0.1,       # Dropout on LoRA activations
    bias="none",            # Don't train bias terms
    task_type="CAUSAL_LM",  # Task type for automatic layer selection logic
    # For Llama-3, target all attention projections:
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

In [5]:
def print_trainable_parameters(model):
    """Report the number of trainable vs total parameters."""
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    print(
        f"Trainable: {trainable_params:,}  |  "
        f"Total: {all_params:,}  |  "
        f"Trainable %: {100 * trainable_params / all_params:.4f}%"
    )

In [6]:
# Inspect the model architecture before LoRA
print("=== Base model architecture (excerpt) ===")
print(model)

=== Base model architecture (excerpt) ===
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), ep

In [7]:
# Wrap the quantised model with LoRA adapters
lora_model = get_peft_model(model, peft_config)

print("=== LoRA-wrapped model ===")
print(lora_model)
print()
print_trainable_parameters(lora_model)

=== LoRA-wrapped model ===
PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
 

Notice that:
- The original `Linear4bit` layers are unchanged and frozen
- Each targeted layer now has additional `lora_A` and `lora_B` sub-modules
- Only ~0.5–2% of parameters are trainable, drastically reducing memory and compute

## Configuring the Trainer

We use TRL's `SFTTrainer` with the modern `SFTConfig` (previously, training arguments were passed directly to `SFTTrainer`, which is now deprecated).

### Key Training Hyperparameters

| Parameter | Value | Explanation |
|---|---|---|
| `per_device_train_batch_size` | 1 | Single sample per GPU forward pass (memory constraint) |
| `gradient_accumulation_steps` | 4 | Effective batch size = 1×4 = 4 |
| `optim` | `paged_adamw_8bit` | AdamW with 8-bit optimizer states; paging offloads to CPU when GPU is full |
| `learning_rate` | 2e-4 | Typical for LoRA fine-tuning |
| `warmup_steps` | 20 | Linearly ramp the LR for the first 20 steps |
| `max_steps` | 500 | Total training steps (use `num_train_epochs` for epoch-based training) |
| `fp16` / `bf16` | bf16 | Mixed-precision training |
| `max_seq_length` | 512 | Truncate inputs longer than this |

### Gradient Accumulation

When batch size 1 is too small for stable training, **gradient accumulation** simulates a larger batch: gradients are summed over multiple forward passes before a single optimizer step. This is memory-free (no extra activations stored), only slightly slower.

### Paged Optimizer

The `paged_adamw_8bit` optimizer from bitsandbytes stores Adam's first and second moment buffers in 8-bit, and pages them to CPU RAM when GPU memory is full. This allows training models that would otherwise OOM.

In [8]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="./results",
    # Batch size & gradient accumulation
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,          # Effective batch size = 4
    # Optimizer
    optim="paged_adamw_8bit",               # Memory-efficient AdamW
    # Learning rate schedule
    learning_rate=2e-4,
    lr_scheduler_type="cosine",             # Cosine decay (often better than constant)
    warmup_steps=20,                        # Gradual LR ramp at start
    # Training length
    max_steps=500,
    # Precision
    bf16=True,                              # Use bfloat16 (preferred over fp16 on modern GPUs)
    # Regularisation
    max_grad_norm=0.3,                      # Gradient clipping
    # Logging & saving
    logging_steps=10,
    save_steps=100,
    report_to="none",                       # Disable wandb/tensorboard for this demo
    # Dataset
    dataset_text_field="text",
    # max_seq_length=512,
)

### Creating the SFTTrainer

`SFTTrainer` is a thin wrapper around `Trainer` that adds:
- Automatic instruction formatting (packing, chat template application)
- Integration with PEFT configs (automatically wraps the model in LoRA if `peft_config` is provided)
- Sequence packing for improved GPU utilisation

In [9]:
from trl import SFTTrainer

# lora_model is already wrapped with LoRA from get_peft_model() above.
# Passing peft_config here would wrap it a second time — omit it.
trainer = SFTTrainer(
    model=lora_model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=sft_config,
)

## Training

Call `trainer.train()` to start. During training, watch the loss — it should decrease steadily. A typical training loss curve for SFT starts around 2.0–3.0 and converges to 0.5–1.5 depending on dataset and task complexity.

**Signs of problems:**
- Loss plateaus immediately → learning rate too low, or dataset formatting issue
- Loss explodes → learning rate too high, or gradient clipping too loose
- Loss reaches near-zero quickly → overfitting (dataset too small or too many steps)

In [10]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


Step,Training Loss
10,1.702032
20,1.415820
30,1.406924
40,1.316258
50,1.538465
60,1.219222
70,1.310221
80,1.402154
90,1.302656
100,1.348147


/home/azagar/Documents/NLP-Course-Tutorials/.vllm/lib/python3.12/site-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error [Errno 104] Connection reset by peer - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3-8B-Instruct.
  warnings.warn(
/home/azagar/Documents/NLP-Course-Tutorials/.vllm/lib/python3.12/site-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in meta-llama/Meta-Llama-3-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=500, training_loss=1.33354638671875, metrics={'train_runtime': 2041.0523, 'train_samples_per_second': 0.98, 'train_steps_per_second': 0.245, 'total_flos': 3.277752565928755e+16, 'train_loss': 1.33354638671875})

## Saving the Adapter

A key advantage of LoRA is that you **only need to save the small adapter** (the $A$ and $B$ matrices), not the full model. This makes it easy to:
- Share adapters with collaborators (a few hundred MB instead of tens of GB)
- Swap adapters on top of the same base model for different tasks
- Publish adapters to the HuggingFace Hub

![LoRA Matrix](LoraMatrix.png)

In [11]:
# Save only the LoRA adapter weights (NOT the full base model)
model_to_save = trainer.model.module if hasattr(trainer.model, "module") else trainer.model
model_to_save.save_pretrained("outputs-spark")

print("Adapter saved to ./outputs-spark")

Adapter saved to ./outputs-spark


## Using the Fine-Tuned Model

### Same session
After training, `trainer.model` is already the PEFT-wrapped fine-tuned model — use it directly.

### New session (loading from disk)
If you are starting a fresh Python process, reload the base model and apply the saved adapter with `PeftModel.from_pretrained`. Do **not** call `get_peft_model()` again — that wraps an already-wrapped model and creates duplicate adapters.

In [12]:
from peft import PeftModel

# --- Same session ---
# trainer.model is already fine-tuned; use it directly.
finetuned_model = trainer.model

# --- New session (run this block instead if starting fresh) ---
# run the code cell below.


In [13]:
# --- New session (run this block instead if starting fresh) ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Store weights in 4-bit
    bnb_4bit_quant_type="nf4",      # Use NormalFloat 4-bit (optimal for normally-distributed weights)
    bnb_4bit_use_double_quant=True, # Quantise the quantisation constants too (~0.37 bits saved/param)
    bnb_4bit_compute_dtype=torch.bfloat16,  # Upcast to bfloat16 for actual matrix multiplications
)

base = AutoModelForCausalLM.from_pretrained(
     model_name, quantization_config=bnb_config, device_map='auto'
     )

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Llama-3 does not have a dedicated pad token — use eos_token as pad
# padding_side='right' avoids overflow issues in half-precision training
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

from peft import PeftModel
finetuned_model = PeftModel.from_pretrained(base, 'outputs')

from datasets import load_dataset

dataset_name = "timdettmers/openassistant-guanaco"
dataset = load_dataset(dataset_name, split="train")

print(f"Dataset size: {len(dataset)} examples")
print(f"Features: {dataset.features}")
print("\nFirst example:")
print(dataset["text"][0])

Loading weights: 100%|██████████| 291/291 [01:41<00:00,  2.87it/s]
Repo card metadata block was not found. Setting CardData to empty.


Dataset size: 9846 examples
Features: {'text': Value('string')}

First example:
### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.### Assistant: "Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees. The presence of a monopsony can result in lower wages and reduced employment opportunities for workers, as the employer has little incentive to increase wages or provide better working conditions.

Recent research has identified potential monopsonies in industries such as retail and fast food, where a few large companies control a significant portion of the market (Bivens & Mishel, 2013). In these industries, worke

In [14]:
# Test the fine-tuned model on a sample prompt from the dataset
prompt = dataset["text"][0].split("### Assistant:")[0] + "### Assistant:"
print("Prompt:")
print(prompt)

device = "cuda:0"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = finetuned_model.generate(**inputs, max_new_tokens=300, do_sample=True, temperature=0.7)

print("\nGenerated:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt:
### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.### Assistant:

Generated:
### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.### Assistant: A monopsony is a market structure in which there is only one buyer of a particular good or service. In the context of the labor market, a monopsony refers to a situation in which there is only one employer of a particular type of labor. This can occur when a single firm has a significant amount of market power and is able to dictate the terms of employment for a particular type of labor.

One example of a potential monopsony in the labor market is the case of nurses. In many countries, there is a shortage of nurses, which gives employ

### Prompting the Fine-Tuned Model with a Custom Input

The Guanaco dataset uses the `### Human: ... ### Assistant:` template. To prompt the fine-tuned model correctly, we format our own question in the same template and let the model complete the `### Assistant:` part.

In [15]:
# Edit this prompt freely — just keep the ### Human / ### Assistant structure
user_prompt = "Explain the difference between supervised and unsupervised learning."

# Format using the Guanaco template
prompt = f"### Human: {user_prompt}\n### Assistant:"

device = "cuda:0"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = finetuned_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.7,
        repetition_penalty=1.1,
    )

# Decode and strip the prompt from the output
full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = full_output[len(prompt):].strip()
print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Supervised learning is a type of machine learning where the model is trained on labeled data. This means that each sample in the training set has been annotated with its corresponding class label or target output, which the model uses to learn the mapping from inputs to outputs.

In contrast, unsupervised learning is a type of machine learning where the model is trained on unlabeled data. There are no target outputs or class labels for the samples in the training set, so the model must find patterns or relationships within the data without any prior knowledge of what it should be looking for.

Supervised learning can be used when there is a clear goal or objective that you want to achieve, such as predicting whether someone will buy a product based on their demographics. In this case, you would have a large dataset of people with demographic information and a target variable indicating whether they bought the product or not. You could then use supervised learning to train a model that 

# Merge model and adapter

In [16]:
from peft import PeftModel

# Load base model in full precision for merging.
# Use device_map={'': 0} (pin to GPU 0) instead of 'auto' — device_map='auto'
# triggers a bug in accelerate's get_balanced_memory when used with PeftModel.
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map={'': 0},
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Apply the saved adapter on top of the full-precision base
peft_model = PeftModel.from_pretrained(base_model, 'outputs', device_map={'': 0})

# Merge adapter weights into the base model and remove adapter modules
merged_model = peft_model.merge_and_unload()

# Save as a standard HuggingFace model (no adapter infrastructure)
merged_model.save_pretrained('outputs_merged')
tokenizer.save_pretrained('outputs_merged')

print('Merged model saved to ./outputs_merged')

`torch_dtype` is deprecated! Use `dtype` instead!
Writing model shards: 100%|██████████| 1/1 [01:10<00:00, 70.85s/it]

Merged model saved to ./outputs_merged


## Summary

In this notebook we covered:

### Fine-Tuning
| Concept | Key idea |
|---|---|
| QLoRA | 4-bit quantisation + LoRA for memory-efficient fine-tuning |
| LoRA | Train only low-rank update matrices; freeze original weights |
| SFTTrainer | HuggingFace TRL trainer for instruction fine-tuning |
| Adapter saving | Save only the small adapter, not the full model |
| Merging | Fold the adapter back into base weights for faster inference |

---

## References

- **QLoRA** — Dettmers et al. (2023): https://arxiv.org/abs/2305.14314
- **LoRA** — Hu et al. (2021): https://arxiv.org/abs/2106.09685
- **4-bit quantisation blog** — HuggingFace: https://huggingface.co/blog/4bit-transformers-bitsandbytes
- **PEFT library** — https://huggingface.co/docs/peft/en/index
- **TRL SFTTrainer** — https://huggingface.co/docs/trl/main/en/sft_trainer